                                 Import Libraries

In [12]:
from langchain_community.document_loaders import PyPDFLoader   #--------- Import PyPDFLoader to load PDF files
print("Success!")

Success!


In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter #---------- Text splitter for chunking documents

from langchain_community.vectorstores import FAISS                 #---------- Vector database (FAISS) for storing embeddings

from langchain_huggingface import HuggingFaceEmbeddings            #---------- Hugging Face embeddings (latest recommended method)

from langchain_ollama import ChatOllama                #------------ Local LLM via Ollama (no API key required)

from langchain_classic.chains import RetrievalQA     # RAG chain (classic version for LangChain 1.0 compatibility)

                           Taking The Path Of Pdf File

In [14]:
pdf_path = r"D:\pythoncode\Gitcode\project3\NationalDietaryGuidelinesforBangladesh-23Aug2025.pdf" 


loader = PyPDFLoader(pdf_path)
documents = loader.load()
print("✅ PDF has been loaded.Total page number:", len(documents))


✅ PDF has been loaded.Total page number: 142


                                Extracts PDF page text.

In [15]:
#Opens a PDF using pdfplumber,, and prints the total character count along with the first 500 characters.
import pdfplumber
with pdfplumber.open(pdf_path) as pdf:  
    first_page = pdf.pages[20]         # Get page 21 (index 20 means page 21)
    text = first_page.extract_text()   # extracts text from page 21  
    print("Text length:", len(text))    # Print total character count
    print(text[:500])                    # Print first 500 characters as a sample

Text length: 2360
MESSAGE
Country Representative, Bangladesh
Food and Agriculture Organization of the United Nations (FAO)
FAO, in collaboration with the World Health Organization, has been providing technical support to
member countries towards the development and implementation of national dietary guidelines
ever since the first International Conference on Nutrition which was held in Rome, Italy, in 1992.
Food-based dietary guidelines (also known as dietary guidelines) are intended to establish a
basis for poli


                   Splits PDF text into smaller chunks, shows the total count, and prints a preview of the first chunk.


In [16]:
#from langchain_text_splitters import RecursiveCharacterTextSplitter
#Importing text splitter from LangChain (library import)


# Create text splitter instance
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,                   # Max chunk size: 500 characters
    chunk_overlap=50,                 # Overlap: 50 chars (preserve context at boundaries)
 
    separators=["\n\n", "\n", " ", ""] # Split priority: paragraphs → lines → spaces → any      

)

# Split loaded PDF documents into chunks    
chunks = text_splitter.split_documents(documents)


# Print total number of chunks created
print(f"The total number of  created chunk: {len(chunks)} ")

# Print first 300 chars of first chunk as sample                                            
print("\nSample of the first chank:\n", chunks[0].page_content[:300])

The total number of  created chunk: 1146 

Sample of the first chank:
 NA TIONAL DIET ARY GUIDELINES
FOR BANGLADESH
NA TIONAL DIET ARY GUIDELINES
FOR BANGLADESH



                                               Creates vector database.


In [17]:

# 1.Load a small & fast embedding model 
#from langchain_huggingface import HuggingFaceEmbeddings  


embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 2.Use FAISS as vector database 
#from langchain_community.vectorstores import FAISS



vectorstore = FAISS.from_documents(chunks, embeddings)    #Convert chunks to embeddings & store in FAISS


#3.Save vector database to disk (for future reuse)
vectorstore.save_local("faiss_index")

#4. Print success message
print("✅ Vector Database has been created and saved ")

Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 3105.71it/s]


✅ Vector Database has been created and saved 


                                                   Loading Vector Database, Creates Retriever.

In [18]:

#from langchain_community.vectorstores import FAISS
#from langchain_huggingface import HuggingFaceEmbeddings  


                 # Load embedding model (converts text to numbers)
                # "all-MiniLM-L6-v2" is a small, fast model

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


                   #Load previously saved vector database from disk
                   # Reading from "faiss_index" folder
                   # allow_dangerous_deserialization=True : assuming local file is safe

vectorstore = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)

                     # Create retriever (search engine for finding matches)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})  # Will find top 3 most similar chunks to the user's question

print("✅ Retriever is ready!")

Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 3338.18it/s]


✅ Retriever is ready!


                                  Creates RAG chain.

In [19]:


# 3. Load local LLM (no API key needed, works offline)
llm = ChatOllama(model="llama3.2:1b", temperature=0.3)

# 4. Create RAG chain (combines retriever + LLM)
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True

)


                                        TESTING:
    Asks a question to the RAG chain, prints the answer, and shows the first 300 characters of each source       chunk used.
                                                     

In [20]:
# 1. Ask a question

question = "what food is suitable for diabatis patient?"  #------- can be changed based on  given pdf
response = qa_chain.invoke(question)     #--------------- Get response from the RAG chain


#  2. Display answer and sources

print("⭐Answer", response['result'])
print("\n📄 Source documents (where the answer came from):")

# Loop through each source chunk
for i, doc in enumerate(response['source_documents']):
    print(f"\n--- Part {i+1} ---")
    print(doc.page_content[:300])  #---------------------Print first 300 chars of each source

⭐Answer For a diabetic patient, it's essential to focus on nutrient-dense foods that are low in added sugars, refined carbohydrates, and saturated fats. Here are some suitable food options:

**Fruits:**

* Fresh fruits like berries, citrus fruits, apples, bananas, and pears
* Dried fruits like dates, apricots, prunes, and raisins (in moderation)

**Vegetables:**

* Dark leafy greens like spinach, kale, broccoli, and collard greens
* Cruciferous vegetables like cauliflower, Brussels sprouts, and carrots
* Allium vegetables like onions, garlic, and shallots
* Root vegetables like sweet potatoes, beets, and parsnips

**Protein sources:**

* Lean meats like chicken, turkey, fish (in moderation), and pork
* Legumes like lentils, chickpeas, black beans, and kidney beans
* Nuts and seeds like almonds, walnuts, chia seeds, and flaxseeds
* Low-fat dairy products like milk, yogurt, cheese, and eggs

**Whole grains:**

* Brown rice, quinoa, whole wheat bread, and whole grain pasta
* Oats, barley,

                                      2      

In [21]:
# 1. Interactive chat loop

print("\n\n⭐⭐⭐⭐⭐⭐ Chatbot is ready!⭐⭐⭐⭐⭐⭐ (Type 'exit' to quit)\n")

while True:

    
    user_question = input("\n❓❓❓Your Question: ")    #-------- Get user input
    

    # Check if user wants to exit
    if user_question.lower() == 'exit':
        print("⚡⚡⚡Chatbot stopped. Thank you!")
        break

    
     
    response = qa_chain.invoke(user_question)     #------- Get response from RAG chain

    
   
    print("🌸🌸🌸 Answer:", response['result'])    # ------Print the answer

    
    
    print("-" * 50)            # ------Print a separator line



⭐⭐⭐⭐⭐⭐ Chatbot is ready!⭐⭐⭐⭐⭐⭐ (Type 'exit' to quit)




❓❓❓Your Question:  How much water should an adult drink daily according to the guidelines?


🌸🌸🌸 Answer: According to the guidelines, an adult should drink at least 10 to 12 glasses of fluid per day.
--------------------------------------------------



❓❓❓Your Question:  exit


⚡⚡⚡Chatbot stopped. Thank you!
